Сравнение SVM, Fuzzy k‑NN и AutoML для классификации нарушений сердечного ритма по ЭКГ
Датасет: PTB-XL (PhysioNet)

Выполнил: студент гр. 737-01 Беликов Андрей Юрьевич
Руководители: Андриков Д.А., Андриянова Е.В.


In [ ]:
!wget -r -N -c -np https://physionet.org/files/ptb-xl/1.0.3/

In [ ]:
!pip install flaml wfdb neurokit2 tqdm

In [ ]:
from flaml import AutoML

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
import wfdb
import neurokit2 as nk
import ast
from tqdm import tqdm
from pathlib import Path
from collections import Counter
from scipy.spatial.distance import cdist
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

In [ ]:
# Подавление предупреждений для чистоты вывода
warnings.filterwarnings('ignore')

# Константы
RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15

# Настройка графиков
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style="whitegrid")

print("=" * 70)
print("СРАВНЕНИЕ SVM, FUZZY k-NN И AUTOML ДЛЯ КЛАССИФИКАЦИИ ЭКГ")
print("=" * 70)

In [ ]:
class FuzzyKNNClassifier:
    """
    Нечёткий k-ближайших соседей (Fuzzy k-NN).

    В отличие от обычного k-NN, который выдаёт жёсткий класс,
    fuzzy версия возвращает вероятность принадлежности к каждому классу
    на основе взвешенного голосования ближайших соседей.

    Параметры:
    -----------
    n_neighbors : int
        Количество ближайших соседей (k)
    m : float
        Параметр нечёткости (обычно 2.0)
    """

    def __init__(self, n_neighbors: int = 7, m: float = 2.0):
        self.n_neighbors = n_neighbors
        self.m = m
        self.X_train = None
        self.y_train = None
        self.classes_ = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'FuzzyKNNClassifier':
        """Запоминает обучающую выборку."""
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        self.classes_ = np.unique(y)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """
        Возвращает вероятности принадлежности к классам.

        Алгоритм:
        1. Для каждого тестового объекта находим k ближайших соседей
        2. Каждому соседу присваиваем вес = 1 / (distance ** (2/(m-1)))
        3. Вероятность класса = сумма весов соседей этого класса / сумма всех весов
        """
        X = np.asarray(X, dtype=float)
        n_classes = len(self.classes_)
        probs = np.zeros((len(X), n_classes))
        eps = 1e-9
        exponent = 2.0 / (self.m - 1.0) if self.m > 1 else 2.0

        for i, x in enumerate(X):
            # Евклидовы расстояния до всех обучающих объектов
            distances = cdist([x], self.X_train, metric='euclidean')[0]
            # Находим k ближайших соседей
            k = min(self.n_neighbors, len(self.X_train))
            idx = np.argsort(distances)[:k]
            dists = distances[idx] + eps
            # Веса соседей (обратно пропорциональны расстоянию)
            weights = 1.0 / (dists ** exponent)

            # Взвешенное голосование
            for cls_idx, cls in enumerate(self.classes_):
                mask = self.y_train[idx] == cls
                probs[i, cls_idx] = np.sum(weights[mask]) / np.sum(weights)

        # Нормализация (сумма вероятностей = 1)
        probs = probs / probs.sum(axis=1, keepdims=True)
        return probs

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Возвращает класс с максимальной вероятностью."""
        probs = self.predict_proba(X)
        return self.classes_[np.argmax(probs, axis=1)]


print("\n[1/8] Класс FuzzyKNNClassifier определён")

In [ ]:
def load_ptbxl_features() -> tuple[np.ndarray, np.ndarray]:
    """
    Датасет PTB-XL, извлекаем признаки с помощью NeuroKit2.
    """
    data_path = Path("ptb-xl")
    if not data_path.exists():
        raise FileNotFoundError(
            "Папка с датасетом PTB-XL не найдена. "
        )

    metadata = pd.read_csv(data_path / "ptbxl_database.csv", index_col='ecg_id')
    print(f"  Загружено {len(metadata)} записей.")

    scp_statements = pd.read_csv(data_path / "scp_statements.csv", index_col=0)

    diagnostic_superclass = {
        'NORM': 'NORM', 'CD': 'CD', 'HYP': 'HYP',
        'MI': 'MI', 'STTC': 'STTC'
    }

    def get_superclass(diag):
        for key in diagnostic_superclass:
            if key in diag:
                return diagnostic_superclass[key]
        return None

    scp_to_superclass = {
        idx: get_superclass(row['description'])
        for idx, row in scp_statements.iterrows()
    }

    labels = []
    for _, row in metadata.iterrows():
        scp_codes = ast.literal_eval(row['scp_codes'])
        # Выбираем 'diagnostic' классы с высшим приоритетом
        present_diags = [
            code for code, prob in scp_codes.items()
            if scp_statements.loc[code, 'diagnostic_class'] == 1
        ]
        superclasses = [scp_to_superclass[code] for code in present_diags if scp_to_superclass[code]]
        if superclasses:
            labels.append(superclasses[0])
        else:
            labels.append('NORM')

    class_names = ['NORM', 'CD', 'HYP', 'MI', 'STTC']
    y = np.array([class_names.index(lbl) for lbl in labels])

    feature_list = []
    for ecg_id in tqdm(metadata.index, desc="  Извлечение признаков"):
        # Загружаем сигнал
        signal, _ = wfdb.rdsamp(str(data_path / metadata.loc[ecg_id, 'filename_lr']))
        # Приводим к 12-ти отведениям, если их меньше
        if signal.shape[1] < 12:
            signal = np.pad(signal, ((0,0), (0,12-signal.shape[1])))
        else:
            signal = signal[:, :12]
        # Берём первое отведение для анализа (или можно усреднить)
        ecg_signal = signal[:, 0]
        # Очищаем сигнал
        cleaned = nk.ecg_clean(ecg_signal, sampling_rate=500)
        # Находим R-пики
        _, rpeaks = nk.ecg_peaks(cleaned, sampling_rate=500)
        # Извлекаем 10 статистик по RR-интервалам
        features = nk.hrv_time(rpeaks, sampling_rate=500, show=False)
        # Собираем все признаки в вектор
        feature_vector = [
            features['HRV_RMSSD'].iloc[0], features['HRV_MeanNN'].iloc[0],
            features['HRV_SDNN'].iloc[0], features['HRV_CVS'].iloc[0],
            features['HRV_pNN50'].iloc[0], features['HRV_MADNN'].iloc[0],
            features['HRV_MCVNN'].iloc[0], features['HRV_IQRNN'].iloc[0],
            features['HRV_HTI'].iloc[0], features['HRV_ShanEn'].iloc[0]
        ]
        feature_list.append(feature_vector)

    X = np.array(feature_list)
    return X, y

In [ ]:
# ============================================================================
# 3. РАЗВЕДОЧНЫЙ АНАЛИЗ ДАННЫХ (EDA)
# ============================================================================

print("\n[3/8] Разведочный анализ данных (EDA)...")

# Создаём DataFrame для удобства
feature_names = [f'feat_{i}' for i in range(X.shape[1])]
df_eda = pd.DataFrame(X, columns=feature_names)
df_eda['target'] = y
df_eda['target_name'] = df_eda['target'].map(CLASS_NAMES)

# 3.1 Распределение классов
print("\n  Распределение целевых классов:")
for cls_name, count in df_eda['target_name'].value_counts().items():
    print(f"    {cls_name}: {count} ({count/len(df_eda)*100:.1f}%)")

# 3.2 Статистика признаков
print("\n  Статистика признаков (первые 5):")
print(df_eda[feature_names[:5]].describe().round(3))

# 3.3 Проверка пропусков
missing_counts = df_eda.isna().sum()
if missing_counts.sum() > 0:
    print(f"\n  Обнаружены пропуски: {missing_counts.sum()} ячеек")
else:
    print("\n  Пропуски отсутствуют")

# 3.4 Визуализация (сохраняем графики)
output_dir = Path("experiment_results")
output_dir.mkdir(exist_ok=True, parents=True)

# График распределения классов
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

class_counts = df_eda['target_name'].value_counts()
axes[0].bar(class_counts.index, class_counts.values, color='#2253A1', alpha=0.8)
axes[0].set_title('Распределение классов', fontsize=12)
axes[0].set_ylabel('Количество записей')
axes[0].tick_params(axis='x', rotation=45)

# PCA визуализация
scaler_temp = StandardScaler()
X_scaled_temp = scaler_temp.fit_transform(X)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled_temp)

colors = ['#2253A1', '#5B8FD6', '#7BAFD4', '#AAC7E8', '#0E2D69']
for i, cls in enumerate(sorted(np.unique(y))):
    mask = y == cls
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors[i], label=CLASS_NAMES[cls], alpha=0.7, s=30)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('PCA визуализация данных')
axes[1].legend(loc='best', fontsize=8)

plt.tight_layout()
plt.savefig(output_dir / 'eda_plots.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"\n  Графики сохранены в {output_dir / 'eda_plots.png'}")

In [ ]:
# ============================================================================
# 4. РАЗДЕЛЕНИЕ ДАННЫХ НА TRAIN / VAL / TEST
# ============================================================================

print("\n[4/8] Разделение данных на train/val/test...")

# Первое разбиение: train+val (85%) и test (15%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

# Второе разбиение: из train+val выделяем validation (15% от всего датасета)
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_ratio,
    stratify=y_train_val, random_state=RANDOM_STATE
)

print(f"  Обучающая выборка: {X_train.shape[0]} записей")
print(f"  Валидационная выборка: {X_val.shape[0]} записей")
print(f"  Тестовая выборка: {X_test.shape[0]} записей")

print("\n  Распределение классов в обучающей выборке:")
for cls, count in Counter(y_train).items():
    print(f"    {CLASS_NAMES[cls]}: {count} ({count/len(y_train)*100:.1f}%)")

In [ ]:
# ============================================================================
# 5. МАСШТАБИРОВАНИЕ ПРИЗНАКОВ
# ============================================================================

print("\n[5/8] Масштабирование признаков (StandardScaler)...")
print("  (SVM и k-NN чувствительны к масштабу признаков)")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"  Средние после масштабирования: {X_train_scaled.mean(axis=0)[:3]}...")
print(f"  Стандартные отклонения: {X_train_scaled.std(axis=0)[:3]}...")

In [ ]:
# ============================================================================
# 6. ОБУЧЕНИЕ SVM
# ============================================================================

print("\n[6/8] Обучение SVM с подбором гиперпараметров...")

# Создаем pipeline для SVM
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),  # дополнительная страховка
    ('svm', SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE))
])

# Сетка гиперпараметров для GridSearch
param_grid_svm = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 0.1, 0.01, 0.001]
}

# 5-кратная стратифицированная кросс-валидация
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("  Запуск GridSearchCV (может занять некоторое время)...")
start_time = time.perf_counter()

svm_grid = GridSearchCV(
    svm_pipeline, param_grid_svm, cv=cv, scoring='roc_auc_ovr',
    n_jobs=-1, verbose=0
)
svm_grid.fit(X_train_scaled, y_train)

svm_fit_time = time.perf_counter() - start_time

print(f"\n  Лучшие параметры: {svm_grid.best_params_}")
print(f"  Лучший CV ROC-AUC: {svm_grid.best_score_:.4f}")
print(f"  Время обучения: {svm_fit_time:.2f} сек")

# Предсказания на валидационной выборке
y_pred_svm_val = svm_grid.predict(X_val_scaled)
y_proba_svm_val = svm_grid.predict_proba(X_val_scaled)

print("\n  Метрики на валидационной выборке:")
print(f"    Accuracy: {accuracy_score(y_val, y_pred_svm_val):.4f}")
print(f"    Macro F1: {f1_score(y_val, y_pred_svm_val, average='macro'):.4f}")
print(f"    ROC-AUC: {roc_auc_score(y_val, y_proba_svm_val, multi_class='ovr', average='macro'):.4f}")


In [ ]:
# ============================================================================
# 7. ОБУЧЕНИЕ FUZZY K-NN
# ============================================================================

print("\n[7/8] Обучение Fuzzy k-NN с подбором гиперпараметров...")

param_grid_fuzzy = {
    'n_neighbors': [5, 7, 9, 11, 15],
    'm': [1.5, 2.0, 2.5]
}

best_f1 = -1
best_params_fuzzy = None
best_model_fuzzy = None

print("  Перебор параметров:")
total_combinations = len(param_grid_fuzzy['n_neighbors']) * len(param_grid_fuzzy['m'])
current = 0

start_time = time.perf_counter()

for k in param_grid_fuzzy['n_neighbors']:
    for m in param_grid_fuzzy['m']:
        current += 1
        print(f"    [{current}/{total_combinations}] k={k}, m={m}")

        model = FuzzyKNNClassifier(n_neighbors=k, m=m)
        model.fit(X_train_scaled, y_train)
        y_pred_val = model.predict(X_val_scaled)
        score = f1_score(y_val, y_pred_val, average='macro')

        if score > best_f1:
            best_f1 = score
            best_params_fuzzy = {'n_neighbors': k, 'm': m}
            best_model_fuzzy = model

fuzzy_fit_time = time.perf_counter() - start_time

print(f"\n  Лучшие параметры: {best_params_fuzzy}")
print(f"  Validation Macro F1: {best_f1:.4f}")
print(f"  Время обучения: {fuzzy_fit_time:.2f} сек")

# Предсказания на валидационной выборке
y_pred_fuzzy_val = best_model_fuzzy.predict(X_val_scaled)
y_proba_fuzzy_val = best_model_fuzzy.predict_proba(X_val_scaled)

print("\n  Метрики на валидационной выборке:")
print(f"    Accuracy: {accuracy_score(y_val, y_pred_fuzzy_val):.4f}")
print(f"    Macro F1: {f1_score(y_val, y_pred_fuzzy_val, average='macro'):.4f}")
print(f"    ROC-AUC: {roc_auc_score(y_val, y_proba_fuzzy_val, multi_class='ovr', average='macro'):.4f}")

In [ ]:
# ============================================================================
# 8. ОБУЧЕНИЕ AUTOML (FLAML)
# ============================================================================

# Попытка импортировать FLAML
try:
    from flaml import AutoML
    FLAML_AVAILABLE = True
except ImportError:
    FLAML_AVAILABLE = False
    print("\n  FLAML не установлен. AutoML часть будет пропущена.")
    print("  Установите FLAML: pip install flaml")

if FLAML_AVAILABLE:
    print("\n[8/8] Обучение AutoML (FLAML)...")
    print("  Автоматический подбор модели и гиперпараметров")

    start_time = time.perf_counter()

    automl = AutoML()
    automl.fit(
        X_train_scaled, y_train,
        task='classification',
        time_budget=120,  # 2 минуты на поиск
        metric='log_loss',
        seed=RANDOM_STATE,
        verbose=0
    )

    automl_fit_time = time.perf_counter() - start_time

    print(f"\n  Лучшая модель: {automl.best_estimator}")
    print(f"  Лучшие параметры: {automl.best_config}")
    print(f"  Время обучения: {automl_fit_time:.2f} сек")

    # Предсказания на валидационной выборке
    y_pred_automl_val = automl.predict(X_val_scaled)
    y_proba_automl_val = automl.predict_proba(X_val_scaled)

    print("\n  Метрики на валидационной выборке:")
    print(f"    Accuracy: {accuracy_score(y_val, y_pred_automl_val):.4f}")
    print(f"    Macro F1: {f1_score(y_val, y_pred_automl_val, average='macro'):.4f}")
    print(f"    ROC-AUC: {roc_auc_score(y_val, y_proba_automl_val, multi_class='ovr', average='macro'):.4f}")
else:
    automl = None
    automl_fit_time = 0
    print("\n[8/8] AutoML пропущен (FLAML не установлен)")

In [ ]:
# ============================================================================
# 9. ФИНАЛЬНАЯ ОЦЕНКА НА ТЕСТОВОЙ ВЫБОРКЕ
# ============================================================================

print("\n" + "=" * 70)
print("ФИНАЛЬНАЯ ОЦЕНКА НА ТЕСТОВОЙ ВЫБОРКЕ")
print("=" * 70)

# 9.1 Оценка SVM
print("\n--- SVM ---")
y_pred_svm = svm_grid.predict(X_test_scaled)
y_proba_svm = svm_grid.predict_proba(X_test_scaled)

svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_f1_macro = f1_score(y_test, y_pred_svm, average='macro')
svm_f1_weighted = f1_score(y_test, y_pred_svm, average='weighted')
svm_roc_auc = roc_auc_score(y_test, y_proba_svm, multi_class='ovr', average='macro')

print(f"  Accuracy: {svm_accuracy:.4f}")
print(f"  Macro F1: {svm_f1_macro:.4f}")
print(f"  Weighted F1: {svm_f1_weighted:.4f}")
print(f"  ROC-AUC (macro OVR): {svm_roc_auc:.4f}")

# 9.2 Оценка Fuzzy k-NN
print("\n--- Fuzzy k-NN ---")
y_pred_fuzzy = best_model_fuzzy.predict(X_test_scaled)
y_proba_fuzzy = best_model_fuzzy.predict_proba(X_test_scaled)

fuzzy_accuracy = accuracy_score(y_test, y_pred_fuzzy)
fuzzy_f1_macro = f1_score(y_test, y_pred_fuzzy, average='macro')
fuzzy_f1_weighted = f1_score(y_test, y_pred_fuzzy, average='weighted')
fuzzy_roc_auc = roc_auc_score(y_test, y_proba_fuzzy, multi_class='ovr', average='macro')

print(f"  Accuracy: {fuzzy_accuracy:.4f}")
print(f"  Macro F1: {fuzzy_f1_macro:.4f}")
print(f"  Weighted F1: {fuzzy_f1_weighted:.4f}")
print(f"  ROC-AUC (macro OVR): {fuzzy_roc_auc:.4f}")

# 9.3 Оценка AutoML
if FLAML_AVAILABLE and automl is not None:
    print("\n--- AutoML (FLAML) ---")
    y_pred_automl = automl.predict(X_test_scaled)
    y_proba_automl = automl.predict_proba(X_test_scaled)

    automl_accuracy = accuracy_score(y_test, y_pred_automl)
    automl_f1_macro = f1_score(y_test, y_pred_automl, average='macro')
    automl_f1_weighted = f1_score(y_test, y_pred_automl, average='weighted')
    automl_roc_auc = roc_auc_score(y_test, y_proba_automl, multi_class='ovr', average='macro')

    print(f"  Accuracy: {automl_accuracy:.4f}")
    print(f"  Macro F1: {automl_f1_macro:.4f}")
    print(f"  Weighted F1: {automl_f1_weighted:.4f}")
    print(f"  ROC-AUC (macro OVR): {automl_roc_auc:.4f}")

In [ ]:
# ============================================================================
# ДОПОЛНИТЕЛЬНЫЕ ГРАФИКИ (в стиле Моховой, Ковязиной, Седых)
# ============================================================================

print("\n" + "=" * 70)
print("ПОСТРОЕНИЕ ГРАФИКОВ")
print("=" * 70)

# Создаём директорию для графиков
output_dir = Path("experiment_results")
output_dir.mkdir(exist_ok=True, parents=True)


# ----------------------------------------------------------------------------
# ГРАФИК 1: ROC-кривые для всех моделей (как у Моховой и Ковязиной)
# ----------------------------------------------------------------------------
print("\n[1/3] Построение ROC-кривых...")

# Преобразуем в бинарную задачу: норма (класс 0) vs патология (все остальные)
y_test_binary = (y_test != 0).astype(int)

# Получаем вероятности для класса "патология" (1 - вероятность нормы)
y_proba_svm_binary = 1 - y_proba_svm[:, 0]
y_proba_fuzzy_binary = 1 - y_proba_fuzzy[:, 0]

plt.figure(figsize=(8, 7))

# ROC-кривая для SVM
fpr_svm, tpr_svm, _ = roc_curve(y_test_binary, y_proba_svm_binary)
auc_svm_binary = roc_auc_score(y_test_binary, y_proba_svm_binary)
plt.plot(fpr_svm, tpr_svm, label=f'SVM (AUC={auc_svm_binary:.3f})',
         linewidth=2, color='#2253A1')

# ROC-кривая для Fuzzy k-NN
fpr_fuzzy, tpr_fuzzy, _ = roc_curve(y_test_binary, y_proba_fuzzy_binary)
auc_fuzzy_binary = roc_auc_score(y_test_binary, y_proba_fuzzy_binary)
plt.plot(fpr_fuzzy, tpr_fuzzy, label=f'Fuzzy k-NN (AUC={auc_fuzzy_binary:.3f})',
         linewidth=2, color='#5B8FD6')

# ROC-кривая для AutoML (если доступен)
if FLAML_AVAILABLE and automl is not None:
    y_proba_automl_binary = 1 - y_proba_automl[:, 0]
    fpr_automl, tpr_automl, _ = roc_curve(y_test_binary, y_proba_automl_binary)
    auc_automl_binary = roc_auc_score(y_test_binary, y_proba_automl_binary)
    plt.plot(fpr_automl, tpr_automl, label=f'AutoML (AUC={auc_automl_binary:.3f})',
             linewidth=2, color='#0E2D69')

# Диагональная линия (случайный классификатор)
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Случайный (AUC=0.5)')

plt.xlabel('False Positive Rate (1 - Специфичность)', fontsize=12)
plt.ylabel('True Positive Rate (Чувствительность)', fontsize=12)
plt.title('ROC-кривые: норма vs патология', fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  Сохранено: {output_dir / 'roc_curves.png'}")


# ----------------------------------------------------------------------------
# ГРАФИК 2: Матрицы ошибок Confusion Matrices (как у всех одногруппников)
# ----------------------------------------------------------------------------
print("\n[2/3] Построение матриц ошибок...")

# Определяем, сколько моделей у нас есть
models_to_plot = []
if 'y_pred_svm' in dir():
    models_to_plot.append(('SVM', y_pred_svm))
models_to_plot.append(('Fuzzy k-NN', y_pred_fuzzy))
if FLAML_AVAILABLE and automl is not None:
    models_to_plot.append(('AutoML', y_pred_automl))

# Создаём матрицы ошибок рядом
fig, axes = plt.subplots(1, len(models_to_plot), figsize=(5*len(models_to_plot), 4.5))

if len(models_to_plot) == 1:
    axes = [axes]

for ax, (model_name, y_pred_model) in zip(axes, models_to_plot):
    cm = confusion_matrix(y_test, y_pred_model)

    # Визуализация матрицы ошибок с цифрами
    im = ax.imshow(cm, cmap='Blues', interpolation='nearest')
    ax.set_title(f'{model_name}', fontsize=12)
    ax.set_xlabel('Предсказанный класс', fontsize=10)
    ax.set_ylabel('Истинный класс', fontsize=10)

    # Подписываем классы
    class_labels = list(CLASS_NAMES.values())
    ax.set_xticks(np.arange(len(class_labels)))
    ax.set_yticks(np.arange(len(class_labels)))
    ax.set_xticklabels(class_labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(class_labels, fontsize=8)

    # Добавляем цифры в ячейки
    for i in range(len(class_labels)):
        for j in range(len(class_labels)):
            text = ax.text(j, i, cm[i, j],
                          ha="center", va="center",
                          color="white" if cm[i, j] > cm.max() / 2 else "black",
                          fontsize=10)

    # Добавляем цветовую шкабу
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('Матрицы ошибок (Confusion Matrices) на тестовой выборке', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(output_dir / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  Сохранено: {output_dir / 'confusion_matrices.png'}")


# ----------------------------------------------------------------------------
# ГРАФИК 3: Сравнение вероятностей (разброс предсказаний) — как у Моховой
# ----------------------------------------------------------------------------
print("\n[3/3] Построение графика сравнения вероятностей...")

# Определяем, где модели не согласны
y_pred_svm_class = svm_grid.predict(X_test_scaled)
disagreement = (y_pred_svm_class != y_pred_fuzzy)

plt.figure(figsize=(8, 7))

# Раскрашиваем точки: красные — модели не согласны, синие — согласны
colors_dot = ['red' if d else 'blue' for d in disagreement]
plt.scatter(y_proba_svm_binary, y_proba_fuzzy_binary, c=colors_dot, alpha=0.7, s=50)

# Диагональная линия (идеальное совпадение)
plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.7, label='Идеальное совпадение')

# Добавляем пороговые линии (0.5)
plt.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, linewidth=1)
plt.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5, linewidth=1)

plt.xlabel('Вероятность патологии (SVM)', fontsize=12)
plt.ylabel('Вероятность патологии (Fuzzy k-NN)', fontsize=12)
plt.title('Сравнение вероятностей: красные — модели не согласны', fontsize=12)

# Легенда
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue',
           label='Модели согласны', markersize=8),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red',
           label='Модели НЕ согласны', markersize=8),
    Line2D([0], [0], color='k', linestyle='--', linewidth=1.5,
           label='Идеальное совпадение')
]
plt.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / 'probability_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  Сохранено: {output_dir / 'probability_comparison.png'}")

# Выводим статистику о расхождениях
n_disagreements = disagreement.sum()
n_total = len(disagreement)
print(f"\n  Статистика: модели НЕ согласны в {n_disagreements} случаях из {n_total} ({n_disagreements/n_total*100:.1f}%)")

# Находим пациентов с высоким риском по одному методу, но не по другому
if FLAML_AVAILABLE and automl is not None:
    print("\n  Примеры расхождений (первые 5):")
    compare_df = pd.DataFrame({
        'Истинный класс': y_test,
        'SVM прогноз': y_pred_svm_class,
        'Fuzzy прогноз': y_pred_fuzzy,
        'SVM вероятность': y_proba_svm_binary.round(3),
        'Fuzzy вероятность': y_proba_fuzzy_binary.round(3)
    })
    disagreements_df = compare_df[disagreement].head(5)
    if len(disagreements_df) > 0:
        print(disagreements_df.to_string(index=True))


print("\n" + "=" * 70)
print("ГРАФИКИ ПОСТРОЕНЫ")
print("=" * 70)
print(f"Все графики сохранены в папку: {output_dir}/")
print("  - roc_curves.png — ROC-кривые")
print("  - confusion_matrices.png — матрицы ошибок")
print("  - probability_comparison.png — сравнение вероятностей")

In [ ]:
# ============================================================================
# 10. СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ
# ============================================================================

print("\n" + "=" * 70)
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("=" * 70)

# Формируем DataFrame с результатами
results_data = [
    {
        "Модель": "SVM (RBF)",
        "Accuracy": round(svm_accuracy, 4),
        "Macro F1": round(svm_f1_macro, 4),
        "Weighted F1": round(svm_f1_weighted, 4),
        "ROC-AUC": round(svm_roc_auc, 4),
        "Время обучения (сек)": round(svm_fit_time, 2)
    },
    {
        "Модель": "Fuzzy k-NN",
        "Accuracy": round(fuzzy_accuracy, 4),
        "Macro F1": round(fuzzy_f1_macro, 4),
        "Weighted F1": round(fuzzy_f1_weighted, 4),
        "ROC-AUC": round(fuzzy_roc_auc, 4),
        "Время обучения (сек)": round(fuzzy_fit_time, 2)
    }
]

if FLAML_AVAILABLE and automl is not None:
    results_data.append({
        "Модель": f"AutoML (FLAML) - {type(automl.best_estimator).__name__}",
        "Accuracy": round(automl_accuracy, 4),
        "Macro F1": round(automl_f1_macro, 4),
        "Weighted F1": round(automl_f1_weighted, 4),
        "ROC-AUC": round(automl_roc_auc, 4),
        "Время обучения (сек)": round(automl_fit_time, 2)
    })

results_df = pd.DataFrame(results_data)
print(results_df.to_string(index=False))

# Сохраняем таблицу в CSV
results_df.to_csv(output_dir / "results_table.csv", index=False)
print(f"\n  Таблица сохранена в {output_dir / 'results_table.csv'}")

In [ ]:
# ============================================================================
# 11. ПРОВЕРКА ГИПОТЕЗЫ
# ============================================================================

print("\n" + "=" * 70)
print("ПРОВЕРКА НАУЧНОЙ ГИПОТЕЗЫ")
print("=" * 70)

print("Гипотеза: AutoML даст качество не хуже ручных подходов,")
print("         а прирост по ROC-AUC относительно настроенного SVM")
print("         будет не больше 5 процентных пунктов.")

if FLAML_AVAILABLE and automl is not None:
    auc_gain = (automl_roc_auc - svm_roc_auc) * 100
    print(f"\n  SVM ROC-AUC: {svm_roc_auc:.4f}")
    print(f"  AutoML ROC-AUC: {automl_roc_auc:.4f}")
    print(f"  Прирост AutoML к SVM: {auc_gain:.2f} процентных пунктов")

    if auc_gain <= 5:
        print("\n  ✅ ГИПОТЕЗА ПОДТВЕРЖДЕНА: прирост ≤ 5 процентных пунктов")
    else:
        print("\n  ❌ ГИПОТЕЗА НЕ ПОДТВЕРЖДЕНА: прирост > 5 процентных пунктов")
else:
    print("\n  AutoML не запущен, проверка гипотезы отложена.")